<a href="https://colab.research.google.com/github/nukitos25-prog/IT_for_me/blob/main/%D0%9B%D0%B0%D0%B1%D0%BE%D1%80%D0%B0%D1%82%D0%BE%D1%80%D0%BD%D0%B0%D1%8F_%D1%80%D0%B0%D0%B1%D0%BE%D1%82%D0%B0_11.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Лабораторная работа 11. Работа с изображениями. Библиотека Pillow

ФИО: **Стрельцов Никита Андреевич**

Группа: **ИНБ-б-о-26-1(1)**

## Задание

Изучите инструментарий библиотеки Pillow. Спроектируйте свой
собственный фильтр. Продемонстрируйте его работу и поясните назначение
предложенного фильтра.

## Решение

Выполним загрузку файла

In [1]:
!wget https://raw.githubusercontent.com/enikolaev/IT-and-Programming/refs/heads/main/data/pract11/cat.jpeg


--2026-11-26 02:21:56--  https://raw.githubusercontent.com/enikolaev/IT-and-Programming/refs/heads/main/data/pract11/cat.jpeg
Resolving raw.githubusercontent.com (raw.githubusercontent.com)... 185.199.108.133, 185.199.109.133, 185.199.110.133, ...
Connecting to raw.githubusercontent.com (raw.githubusercontent.com)|185.199.108.133|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 93715 (92K) [image/jpeg]
Saving to: ‘cat.jpeg.1’

cat.jpeg.1          100%[===================>]  91.52K  --.-KB/s    in 0.001s  

2026-11-26 02:21:56 (67.8 MB/s) - ‘cat.jpeg.1’ saved [93715/93715]



In [2]:
from PIL import Image, ImageOps, ImageFilter, ImageDraw, ImageEnhance
import random

def vintage_filter(image, sepia_intensity=1.0, vignette_strength=0.7, noise_intensity=0.05):
    """
    Создаёт эффект старинной фотографии.

    :param image: исходное изображение PIL
    :param sepia_intensity: сила сепии (0..1)
    :param vignette_strength: сила виньетки (0..1)
    :param noise_intensity: интенсивность шума (0..1)
    :return: обработанное изображение
    """
    img = image.copy().convert("RGB")

    # 1. Сепия
    if sepia_intensity > 0:
        gray = ImageOps.grayscale(img)
        sepia_img = ImageOps.colorize(gray, black="#704214", white="#C0A080")
        img = Image.blend(img, sepia_img, alpha=sepia_intensity)

    # 2. Виньетка
    if vignette_strength > 0:
        width, height = img.size
        # Маска: белый эллипс в центре, чёрный по краям
        mask = Image.new("L", (width, height), 0)
        draw = ImageDraw.Draw(mask)
        draw.ellipse(
            (width * 0.15, height * 0.15, width * 0.85, height * 0.85),
            fill=255
        )
        # Размываем границы
        mask = mask.filter(ImageFilter.GaussianBlur(radius=width * 0.1))
        # Затемняем края
        black = Image.new("RGB", (width, height), (0, 0, 0))
        vignette_img = Image.composite(img, black, mask)
        img = Image.blend(img, vignette_img, alpha=vignette_strength)

    # 3. Шум (плёночное зерно)
    if noise_intensity > 0:
        noise = Image.effect_noise(img.size, 20).convert("RGB")
        img = Image.blend(img, noise, alpha=noise_intensity)

    # 4. Лёгкое повышение контраста
    enhancer = ImageEnhance.Contrast(img)
    img = enhancer.enhance(1.1)

    return img


# Загрузка исходного изображения
original = Image.open("cat.jpeg")

# Применение фильтра
filtered = vintage_filter(original)

# Сохранение результата
filtered.save("cat_vintage.jpg")

# Отображение исходного и обработанного изображений рядом
def show_side_by_side(img1, img2, title1="Исходное", title2="Винтаж"):
    from PIL import ImageDraw, ImageFont
    w1, h1 = img1.size
    w2, h2 = img2.size
    total_w = w1 + w2 + 20
    max_h = max(h1, h2) + 30
    canvas = Image.new("RGB", (total_w, max_h), (255, 255, 255))
    canvas.paste(img1, (0, 30))
    canvas.paste(img2, (w1 + 20, 30))
    draw = ImageDraw.Draw(canvas)
    draw.text((10, 5), title1, fill=(0, 0, 0))
    draw.text((w1 + 30, 5), title2, fill=(0, 0, 0))
    return canvas

result = show_side_by_side(original, filtered)
result.show()

Назначение: имитация старой выцветшей фотографии. Фильтр придаёт снимку тёплый коричневатый оттенок (сепия), затемняет края (виньетка) и добавляет лёгкое зерно (шум). Такой эффект часто используется для стилизации изображений под ретро, создания атмосферных снимков, оформления открыток и постеров.

# Контрольные вопросы

1. Перечислите функции для трансформации изображения: вращение,
обрезка и т.д.

Pillow предоставляет несколько основных методов для геометрических преобразований изображений:

1) resize(size): Изменяет размер изображения до указанных ширины и высоты (в пикселях). Размер задаётся кортежем (ширина, высота).

2) rotate(angle): Поворачивает изображение на заданный угол (в градусах) против часовой стрелки.

3) transpose(method): Позволяет выполнять различные преобразования, такие как поворот на 90, 180, 270 градусов, а также горизонтальное или вертикальное отражение (например, Image.FLIP_LEFT_RIGHT).

4) crop(box): Обрезает прямоугольную область изображения. Параметр box — это кортеж (left, upper, right, lower), определяющий координаты левого верхнего и правого нижнего углов области обрезки.

5) thumbnail(size): Создаёт уменьшенную копию изображения, сохраняя его пропорции, так чтобы оно поместилось в рамки указанного размера.

2. Поясните назначение метода convert(). Поясните возможные значения
параметра данного метода.

Метод convert() предназначен для преобразования изображения из одного цветового режима (mode) в другой. Это необходимо для изменения цветового пространства, уменьшения количества каналов или приведения изображения к формату, подходящему для конкретной операции.

mode: Это основной параметр, строковый код, указывающий целевой цветовой режим. Каждый режим определяет способ хранения и интерпретации данных пикселей. Вот наиболее распространённые значения:

1) "L": 8-битное изображение в оттенках серого (чёрно-белое).

2) "RGB": 24-битное полноцветное изображение (истинный цвет).

3) "RGBA": 32-битное цветное изображение с альфа-каналом (прозрачностью).

4) "CMYK": 32-битное изображение для цветоделения, используется в полиграфии.

5) "1": 1-битное изображение, только чёрный и белый цвет.

6) "P": 8-битное изображение с палитрой.

7) "YCbCr": 24-битный цветовой формат, используемый в видео.

8) "I": 32-битные целочисленные пиксели (высокая точность).

9) "F": 32-битные пиксели с плавающей точкой.

3. Что такое фильтрация изображений? Приведите примеры практического
применения фильтров.

Фильтрация изображений — это процесс обработки изображения с целью изменения его свойств, таких как удаление шума, подчёркивание границ, размытие или повышение резкости. Это достигается путём применения к каждому пикселю изображения (или его окрестности) определённой математической функции.

Примеры практического применения фильтров:

1) Шумоподавление: Медианный фильтр (ImageFilter.MedianFilter) эффективно удаляет импульсный шум («соль и перец») с изображений, например, при обработке фотографий автомобильных номеров. Усредняющий фильтр также используется для удаления зернистости.

2) Повышение резкости: Фильтр ImageFilter.SHARPEN или ImageFilter.UnsharpMask делает изображение более чётким, подчёркивая границы объектов.

3) Размытие: Гауссово размытие (ImageFilter.GaussianBlur) используется для создания эффекта мягкого фокуса, а также как подготовительный этап перед другими алгоритмами обработки.

4) Выделение контуров и границ: Фильтры ImageFilter.FIND_EDGES и ImageFilter.CONTOUR помогают выделить границы объектов на изображении, что полезно для задач компьютерного зрения и распознавания образов.

5) Стилизация: Фильтр ImageFilter.EMBOSS создаёт эффект барельефа (выпуклого изображения).

4. Поясните формат изображения RGB.

RGB — это аддитивная цветовая модель, в которой любой цвет получается путём смешивания трёх базовых цветов: Red (красный), Green (зелёный) и Blue (синий). Это основной формат для отображения цветных изображений на экранах мониторов, телевизоров и в цифровых камерах.

В формате RGB каждый пиксель изображения состоит из трёх цветовых каналов. Стандартная глубина цвета для каждого канала составляет 8 бит, что даёт 256 возможных уровней интенсивности (от 0 до 255). Таким образом, один пиксель в формате RGB занимает 24 бита (3 канала × 8 бит) и может отображать до 16,7 миллионов различных цветов.

5. Поясните значение терминов: растр, формат изображения, палитра,
глубина, разрешение изображения.

1) Растр: Это способ представления изображения в виде двумерной сетки (матрицы) пикселей. Каждый пиксель имеет свои координаты и определённый цвет. Растровое изображение описывает цвет каждой точки, из которой оно состоит.

2) Формат изображения: Это стандартизированный способ организации и хранения данных изображения в файле. Формат определяет, как кодируется информация о пикселях, какие метаданные (например, разрешение, цветовой профиль) сохраняются и каким алгоритмом сжимается изображение. Примеры: JPEG, PNG, BMP, GIF, TIFF.

3) Палитра: Это набор цветов, которые могут быть использованы для отображения изображения. В палитровых изображениях (режим P) каждый пиксель хранит не полный цвет, а индекс (номер) цвета в палитре. Палитра может содержать, например, 16 или 256 цветов.

4) Глубина цвета: Это количество бит, используемое для кодирования цвета одного пикселя. Глубина цвета определяет максимальное количество различных цветов, которое может быть отображено на изображении (2 в степени глубины). Например, глубина 8 бит даёт 256 цветов, 24 бита — более 16 миллионов.

5) Разрешение изображения: Это характеристика, определяющая степень детализации растрового изображения. Обычно измеряется в количестве пикселей по горизонтали и вертикали (например, 1920×1080) или в количестве точек на дюйм (dpi — dots per inch) при печати или сканировании.

6. Опишите возможности библиотеки Pillow. Приведите примеры методов,
которые могут модифицировать изображения.

Pillow — это мощная и удобная библиотека Python для работы с изображениями. Она является форком оригинальной библиотеки PIL (Python Imaging Library) и поддерживает широкий спектр форматов файлов, включая JPEG, PNG, BMP, GIF, TIFF и другие.

Библиотека предоставляет инструменты для выполнения практически любых операций с изображениями:

1) Открытие, сохранение и создание: Image.open(), Image.save(), Image.new().

2) Геометрические преобразования: resize(), rotate(), crop(), transpose(), thumbnail().

3) Изменение цвета и режима: convert().

4) Улучшение и фильтрация: ImageFilter (например, BLUR, SHARPEN, EMBOSS, FIND_EDGES, GaussianBlur), ImageEnhance (для настройки яркости, контраста, насыщенности).

5) Рисование: ImageDraw позволяет рисовать на изображении линии, фигуры и добавлять текст.

6) Попиксельная обработка: Доступ к пикселям через load() для выполнения произвольных операций над цветом каждого пикселя.

7) Разделение и объединение каналов: split() и merge().

Примеры методов, модифицирующих изображение:

1) Открытие изображения

img = Image.open("example.jpg")

2) Изменение размера

resized_img = img.resize((800, 600))

3) Поворот на 90 градусов

rotated_img = img.rotate(90)

4) Обрезка области (лево, верх, право, низ)

cropped_img = img.crop((100, 100, 400, 400))

5) Преобразование в оттенки серого

gray_img = img.convert("L")

6) Применение размытия по Гауссу

blurred_img = img.filter(ImageFilter.GaussianBlur(radius=2))

7) Повышение резкости

sharpened_img = img.filter(ImageFilter.SHARPEN)

8) Увеличение яркости в 1.5 раза

enhancer = ImageEnhance.Brightness(img)

brightened_img = enhancer.enhance(1.5)